# 04_genre_trends

DML: gold_genre_trends — Genre popularity by month.

In [ ]:
%run ../../tools/config/settings
%run ../../tools/delta/upsert

In [ ]:
dbutils.widgets.text("snapshot_date", "")
snapshot_date = dbutils.widgets.get("snapshot_date")

In [ ]:
from pyspark.sql import functions as F

fct   = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.fct_plays")
bph   = spark.table(f"{CATALOG}.{BRONZE_SCHEMA}.bronze_play_history").select("played_at", "track_id", "artist_ids")
dim_a = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.dim_artists").select("artist_id", "primary_genre")

result = (
    fct
    .join(bph, ["played_at", "track_id"], "left")
    .select("play_id", "track_id", F.explode("artist_ids").alias("artist_id"))
    .join(dim_a, "artist_id", "left")
    .filter(F.col("primary_genre").isNotNull())
    .groupBy("primary_genre")
    .agg(
        F.count("play_id").alias("play_count"),
        F.countDistinct("track_id").alias("unique_tracks"),
    )
    .withColumnRenamed("primary_genre", "genre")
    .withColumn("snapshot_date", F.to_date(F.lit(snapshot_date)))
    .select("snapshot_date", "genre", "play_count", "unique_tracks")
)

upsert_delta(result, f"{CATALOG}.{GOLD_SCHEMA}.gold_genre_trends", ["snapshot_date", "genre"])
display(result.orderBy(F.desc("play_count")))